# Sentiment Analysis: From Binary to Nuanced
## A Comparative Tour of Sentiment Models

This notebook runs the **same set of test sentences** through six increasingly sophisticated levels of sentiment analysis, from simple rule-based lexicons to fine-grained multi-label emotion classifiers.

**All models run on CPU on free Google Colab.**

### Levels
1. **Rule-based / Lexicon** — VADER & TextBlob (no neural network)
2. **Binary classifier** — Positive / Negative (DistilBERT on SST-2)
3. **Ternary classifier** — Positive / Negative / Neutral (Twitter-RoBERTa)
4. **5-class scalar** — Star ratings & intensity labels (BERT & DistilBERT)
5. **7-class emotion** — Ekman basic emotions (DistilRoBERTa)
6. **28-class multi-label emotion** — GoEmotions (RoBERTa)

---

## Setup

Install all dependencies. This takes 1–2 minutes on Colab.

In [ ]:
!pip install -q transformers torch vaderSentiment textblob
!python -m textblob.download_corpora lite

## Test Sentences

These sentences are chosen to reveal differences between models. They include straightforward cases, irony, mixed emotions, negation, and ambiguity.

In [ ]:
TEST_SENTENCES = [
    # Clear positive
    "This is the best movie I have ever seen in my entire life.",
    # Clear negative
    "This was an absolutely terrible waste of my time.",
    # Neutral / factual
    "The meeting has been moved to 3 PM on Thursday.",
    # Sarcasm / irony
    "Oh great, another Monday morning. Just what I needed.",
    # Mixed emotions
    "I'm proud of what we accomplished, but I'm exhausted and a little sad it's over.",
    # Negation trap
    "I wouldn't say the food was bad, but I'm not going back.",
    # Subtle disappointment
    "It was fine, I guess. Not what I was hoping for.",
    # Anticipation / excitement with anxiety
    "I'm so nervous about the presentation tomorrow, but also kind of excited.",
    # Backhanded compliment
    "You're actually much smarter than you look.",
    # Complex literary register
    "There was a quiet dignity in the way she accepted the news, though her hands trembled.",
]

# Short labels for display
SHORT_LABELS = [
    "Clear positive",
    "Clear negative",
    "Neutral / factual",
    "Sarcasm / irony",
    "Mixed emotions",
    "Negation trap",
    "Subtle disappointment",
    "Nervous + excited",
    "Backhanded compliment",
    "Complex literary",
]

print(f"Loaded {len(TEST_SENTENCES)} test sentences.")
for i, (label, sent) in enumerate(zip(SHORT_LABELS, TEST_SENTENCES)):
    print(f"  {i+1}. [{label}] {sent}")

## Helper function

A small utility to display results in a clean table.

In [ ]:
from IPython.display import display, HTML

def show_results(title, rows, columns):
    """Display a list of result rows as an HTML table."""
    html = f"<h3>{title}</h3>"
    html += '<table style="border-collapse:collapse; width:100%; font-size:13px;">'
    html += '<tr style="background:#f0f0f0;">'
    for col in columns:
        html += f'<th style="border:1px solid #ccc; padding:6px 8px; text-align:left;">{col}</th>'
    html += '</tr>'
    for i, row in enumerate(rows):
        bg = '#ffffff' if i % 2 == 0 else '#f9f9f9'
        html += f'<tr style="background:{bg};">'
        for val in row:
            html += f'<td style="border:1px solid #ccc; padding:6px 8px;">{val}</td>'
        html += '</tr>'
    html += '</table><br>'
    display(HTML(html))

---
## Level 1: Rule-Based / Lexicon

No neural network at all. These tools look up words in a hand-built dictionary and apply simple rules (negation, intensifiers, punctuation).

**VADER** returns a compound score from −1 (most negative) to +1 (most positive).  
**TextBlob** returns polarity (−1 to +1) and subjectivity (0 = objective, 1 = subjective).

### What to notice
- How do they handle sarcasm? (Sentence 4)
- How do they handle negation? (Sentence 6)
- Can they distinguish emotions, or only polarity?

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

vader = SentimentIntensityAnalyzer()

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    # VADER
    vs = vader.polarity_scores(sent)
    v_compound = f"{vs['compound']:+.3f}"
    v_verdict = "POS" if vs['compound'] >= 0.05 else ("NEG" if vs['compound'] <= -0.05 else "NEU")

    # TextBlob
    tb = TextBlob(sent)
    tb_pol = f"{tb.sentiment.polarity:+.3f}"
    tb_sub = f"{tb.sentiment.subjectivity:.3f}"

    rows.append([label, v_compound, v_verdict, tb_pol, tb_sub])

show_results(
    "Level 1: Rule-Based (VADER & TextBlob)",
    rows,
    ["Sentence", "VADER compound", "VADER verdict", "TextBlob polarity", "TextBlob subjectivity"]
)

---
## Level 2: Binary Neural Classifier (Positive / Negative)

**Model:** `distilbert-base-uncased-finetuned-sst-2-english` (67M params)  
**Training data:** Stanford Sentiment Treebank (movie reviews)

This model has learned language patterns but can only output two labels. Everything gets forced into POSITIVE or NEGATIVE.

### What to notice
- What happens to the neutral/factual sentence? It *must* pick a side.
- Does the confidence score tell us anything the label doesn't?

In [ ]:
from transformers import pipeline

print("Loading binary sentiment model (DistilBERT SST-2)...")
binary_clf = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1  # CPU
)

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    result = binary_clf(sent, truncation=True)[0]
    rows.append([label, result['label'], f"{result['score']:.4f}"])

show_results(
    "Level 2: Binary Classifier (POSITIVE / NEGATIVE)",
    rows,
    ["Sentence", "Label", "Confidence"]
)

---
## Level 3: Ternary Classifier (Positive / Negative / Neutral)

**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest` (~125M params)  
**Training data:** ~124M tweets, fine-tuned on TweetEval

Adding a third category (neutral) is a big conceptual step. The model can now say "this isn't emotionally charged."

### What to notice
- Does the factual sentence finally get classified as neutral?
- How do the confidence scores spread across three categories?

In [ ]:
print("Loading ternary sentiment model (Twitter-RoBERTa)...")
ternary_clf = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=-1,
    top_k=None  # return all scores
)

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    results = ternary_clf(sent, truncation=True)[0]
    # Sort by score descending
    results = sorted(results, key=lambda x: x['score'], reverse=True)
    top_label = results[0]['label']
    scores_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in results])
    rows.append([label, top_label, scores_str])

show_results(
    "Level 3: Ternary Classifier (positive / negative / neutral)",
    rows,
    ["Sentence", "Top Label", "All Scores"]
)

---
## Level 4: Five-Class Scalar Models

These models introduce *degree* — not just polarity, but intensity.

**Model A:** `nlptown/bert-base-multilingual-uncased-sentiment` (167M params)  
Outputs star ratings: 1 star ★ through 5 stars ★★★★★. Trained on product reviews.

**Model B:** `tabularisai/multilingual-sentiment-analysis` (67M params, DistilBERT)  
Five named intensity levels: Very Negative → Very Positive.

### What to notice
- Does "fine, I guess" (sentence 7) get 3 stars or 2?
- How does each model handle the mixed-emotion sentence?

In [ ]:
print("Loading 5-star model (nlptown BERT)...")
star_clf = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=-1,
    top_k=None
)

print("Loading 5-intensity model (tabularisai DistilBERT)...")
intensity_clf = pipeline(
    "sentiment-analysis",
    model="tabularisai/multilingual-sentiment-analysis",
    device=-1,
    top_k=None
)

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    # Star model
    star_res = star_clf(sent, truncation=True)[0]
    star_res = sorted(star_res, key=lambda x: x['score'], reverse=True)
    star_top = star_res[0]['label']

    # Intensity model
    int_res = intensity_clf(sent, truncation=True)[0]
    int_res = sorted(int_res, key=lambda x: x['score'], reverse=True)
    int_top = int_res[0]['label']
    int_conf = f"{int_res[0]['score']:.3f}"

    rows.append([label, star_top, int_top, int_conf])

show_results(
    "Level 4: Five-Class Scalar Models",
    rows,
    ["Sentence", "Star Rating", "Intensity Label", "Intensity Conf."]
)

---
## Level 5: Emotion Classification (7 Emotions)

**Model:** `j-hartmann/emotion-english-distilroberta-base` (DistilRoBERTa)  
**Categories:** anger, disgust, fear, joy, neutral, sadness, surprise  
Based on Ekman's basic emotions.

This is a conceptual leap: the model no longer measures *how positive/negative* but instead asks *what kind of feeling*.

### What to notice
- The clear negative sentence: is it anger, disgust, or sadness?
- The nervous/excited sentence: does the model favor fear or joy?
- The literary sentence: can it detect sadness without explicitly negative words?

In [ ]:
print("Loading 7-emotion model (j-hartmann DistilRoBERTa)...")
emotion_clf = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=-1,
    top_k=None
)

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    results = emotion_clf(sent, truncation=True)[0]
    results = sorted(results, key=lambda x: x['score'], reverse=True)
    top = results[0]
    runner_up = results[1]
    scores_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in results[:4]])
    rows.append([label, f"{top['label']} ({top['score']:.3f})",
                 f"{runner_up['label']} ({runner_up['score']:.3f})", scores_str])

show_results(
    "Level 5: Emotion Classification (7 Ekman Emotions)",
    rows,
    ["Sentence", "Top Emotion", "Runner-Up", "Top 4 Scores"]
)

---
## Level 6: Fine-Grained Multi-Label Emotion (28 Categories)

**Model:** `SamLowe/roberta-base-go_emotions` (RoBERTa, ~125M params)  
**Categories:** admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral

Crucially, this is **multi-label**: a single sentence can trigger *multiple* emotions simultaneously.

### What to notice
- The mixed-emotion sentence should light up several labels at once.
- The backhanded compliment: does the model detect both approval and annoyance?
- The literary sentence: which subtle emotions does it find?

In [ ]:
print("Loading 28-emotion multi-label model (GoEmotions RoBERTa)...")
go_emotions_clf = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    device=-1,
    top_k=None
)

THRESHOLD = 0.15  # Show emotions above this confidence

rows = []
for label, sent in zip(SHORT_LABELS, TEST_SENTENCES):
    results = go_emotions_clf(sent, truncation=True)[0]
    results = sorted(results, key=lambda x: x['score'], reverse=True)

    # Active emotions (above threshold)
    active = [r for r in results if r['score'] >= THRESHOLD]
    if not active:
        active = [results[0]]  # at least show the top one

    active_str = ", ".join([f"{r['label']} ({r['score']:.3f})" for r in active])
    num_active = len(active)

    rows.append([label, num_active, active_str])

show_results(
    f"Level 6: Multi-Label Emotion (28 categories, threshold ≥ {THRESHOLD})",
    rows,
    ["Sentence", "# Active", "Detected Emotions (score)"]
)

---
## Side-by-Side Summary

This final table puts the top-level output from every model next to each other for each sentence, so you can scan across the columns and see how the same sentence gets interpreted at each level of granularity.

In [ ]:
print("Building comparison table...\n")

summary_rows = []
for i, (label, sent) in enumerate(zip(SHORT_LABELS, TEST_SENTENCES)):
    # VADER
    vs = vader.polarity_scores(sent)
    v_out = f"{vs['compound']:+.2f}"

    # Binary
    b_res = binary_clf(sent, truncation=True)[0]
    b_out = b_res['label'][:3]  # POS or NEG

    # Ternary
    t_res = ternary_clf(sent, truncation=True)[0]
    t_res = sorted(t_res, key=lambda x: x['score'], reverse=True)
    t_out = t_res[0]['label'][:3]

    # Stars
    s_res = star_clf(sent, truncation=True)[0]
    s_res = sorted(s_res, key=lambda x: x['score'], reverse=True)
    s_out = s_res[0]['label']

    # 7-emotion
    e_res = emotion_clf(sent, truncation=True)[0]
    e_res = sorted(e_res, key=lambda x: x['score'], reverse=True)
    e_out = e_res[0]['label']

    # GoEmotions (top 3)
    g_res = go_emotions_clf(sent, truncation=True)[0]
    g_res = sorted(g_res, key=lambda x: x['score'], reverse=True)
    g_top3 = [r['label'] for r in g_res[:3]]
    g_out = ", ".join(g_top3)

    summary_rows.append([label, v_out, b_out, t_out, s_out, e_out, g_out])

show_results(
    "Side-by-Side Comparison: All Models",
    summary_rows,
    ["Sentence", "VADER", "Binary", "Ternary", "Stars", "Emotion (7)", "GoEmotions (top 3)"]
)

---
## Try Your Own Sentences

Use the cell below to test any sentence through all six levels at once.

In [ ]:
def analyze_all(sentence):
    """Run a single sentence through every model and display results."""
    print(f'\nAnalyzing: "{sentence}"\n')
    print("=" * 80)

    # Level 1: VADER
    vs = vader.polarity_scores(sentence)
    print(f"\n🔹 VADER:     compound = {vs['compound']:+.3f}  "
          f"(pos={vs['pos']:.3f}, neu={vs['neu']:.3f}, neg={vs['neg']:.3f})")

    # Level 1: TextBlob
    tb = TextBlob(sentence)
    print(f"🔹 TextBlob:  polarity = {tb.sentiment.polarity:+.3f}, "
          f"subjectivity = {tb.sentiment.subjectivity:.3f}")

    # Level 2: Binary
    b = binary_clf(sentence, truncation=True)[0]
    print(f"\n🔸 Binary:    {b['label']} ({b['score']:.4f})")

    # Level 3: Ternary
    t = ternary_clf(sentence, truncation=True)[0]
    t = sorted(t, key=lambda x: x['score'], reverse=True)
    t_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in t])
    print(f"🔸 Ternary:   {t_str}")

    # Level 4: Stars
    s = star_clf(sentence, truncation=True)[0]
    s = sorted(s, key=lambda x: x['score'], reverse=True)
    s_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in s])
    print(f"🔸 Stars:     {s_str}")

    # Level 5: 7-emotion
    e = emotion_clf(sentence, truncation=True)[0]
    e = sorted(e, key=lambda x: x['score'], reverse=True)
    e_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in e])
    print(f"\n🔷 Emotions:  {e_str}")

    # Level 6: GoEmotions
    g = go_emotions_clf(sentence, truncation=True)[0]
    g = sorted(g, key=lambda x: x['score'], reverse=True)
    active = [r for r in g if r['score'] >= 0.10]
    if not active:
        active = g[:3]
    g_str = ", ".join([f"{r['label']}={r['score']:.3f}" for r in active])
    print(f"🔷 GoEmotions ({len(active)} active): {g_str}")
    print("\n" + "=" * 80)

In [ ]:
# Try it out! Change the sentence below and re-run this cell.
analyze_all("I can't believe she actually did it — I'm so proud of her.")

In [ ]:
# Another example — try literary or ambiguous sentences
analyze_all("He smiled, but there was something broken behind his eyes.")

---
## Discussion Questions

1. Which model would you choose if you were analyzing product reviews at scale? Why?
2. Which sentences caused the biggest *disagreement* between models? What does that tell us about those sentences — or those models?
3. The GoEmotions model can assign multiple labels. When is that an advantage? When might it be a disadvantage?
4. None of these models handle sarcasm well. Why is sarcasm so hard for these approaches? What would a model need to handle it better?
5. The literary sentence ("There was a quiet dignity...") is emotionally complex. Compare how the different models handle it. What gets lost at each level of simplification?
6. Are there kinds of texts (genres, registers, historical periods) where these models would be especially unreliable? Why?